# Demo 02: Judge Bias Detection

**Paper:** [Am I More Pointwise or Pairwise? Revealing Position Bias in Rubric-Based LLM-as-a-Judge](https://arxiv.org/abs/2602.02219) (Feb 2026)

**Key insight:** LLM judges exhibit systematic biases. Position bias causes judges to prefer responses presented first (or last). Verbosity bias causes judges to prefer longer responses regardless of quality. These biases can make your evaluation unreliable.

**What you will learn:**
1. Detect position bias by shuffling response order
2. Detect verbosity bias by padding responses
3. Measure bias magnitude with repeated runs

**Time:** 20 minutes

In [ ]:
# %pip install strands-agents strands-agents-evals boto3

## Step 1: Create two responses with the same quality

**The experiment design:** To detect bias, we need a controlled setup. Two responses that contain the **exact same facts** (same flights, same prices) but differ in **style only** (numbered list vs. bullet points, short vs. verbose).

If the judge scores them differently, the difference cannot be about quality (they have the same facts). It must be about **bias** (the judge prefers one style over another).

| Response | Facts | Style | Length |
|----------|-------|-------|:------:|
| A | BA117 $450, DL1 $520 | Numbered list, concise | ~130 chars |
| B | BA117 $450, DL1 $520 | Bullet points, slightly longer | ~170 chars |

**Expected result if no bias:** Both responses should score the same (around 0.8-0.9).

In [ ]:
import nest_asyncio
nest_asyncio.apply()  # Fix for Jupyter async event loop

from strands_evals import Experiment, Case
from strands_evals.evaluators import OutputEvaluator

MODEL = "gpt-4o-mini"

# Two equally good responses with different styles
RESPONSE_A = (
    "Flights from NYC to London next Friday:\n"
    "1. BA117 - JFK 7PM to LHR 7AM - $450\n"
    "2. DL1 - JFK 9:30PM to LHR 9:30AM - $520"
)

RESPONSE_B = (
    "Here are your options for NYC to London next Friday:\n"
    "- British Airways BA117 departs JFK at 7PM, lands LHR 7AM, costs $450\n"
    "- Delta DL1 departs JFK at 9:30PM, lands LHR 9:30AM, costs $520"
)

RUBRIC = (
    "Rate helpfulness 0-1. A helpful response lists specific flights "
    "with airline, time, and price. Score 0.8+ for complete info."
)

print(f"Response A: {len(RESPONSE_A)} chars")
print(f"Response B: {len(RESPONSE_B)} chars")

## Step 2: Detect position bias

**What is position bias?** The judge gives a different score to the same response depending on what context surrounds it. For example, Response A might score 0.85 alone but 0.75 when presented after Response B.

**How we test it:** We score each response in two contexts:
- **Alone:** Just the response, no other context
- **After the other response:** The response is preceded by the other response in the input

If the score changes based on what came before, position bias is present.

**What to look for in the results:** Compare "A_alone" vs "A_after_B". A difference greater than 0.1 suggests meaningful position bias.

In [ ]:
"""Position bias test: score the same response presented in different contexts."""

evaluator = OutputEvaluator(rubric=RUBRIC, model=MODEL)

# Test: Score Response A when presented alone vs. after Response B
cases_position = [
    Case(name="A_alone", input="Find flights NYC to London"),
    Case(name="A_after_B", input=f"Find flights NYC to London\n\n[Previous response: {RESPONSE_B}]"),
    Case(name="B_alone", input="Find flights NYC to London"),
    Case(name="B_after_A", input=f"Find flights NYC to London\n\n[Previous response: {RESPONSE_A}]"),
]

responses_position = {
    "A_alone": RESPONSE_A,
    "A_after_B": RESPONSE_A,
    "B_alone": RESPONSE_B,
    "B_after_A": RESPONSE_B,
}

experiment = Experiment(cases=cases_position, evaluators=[evaluator])
reports = experiment.run_evaluations(lambda case: responses_position[case.name])
reports[0].display()

# Check for position bias
scores = {case.name: [] for case in cases_position}
for case_result in reports[0].cases:
    scores[case_result["case_name"]] = case_result.get("score", 0)

print("\nPosition bias analysis:")
print(f"  A alone vs A after B: check if scores differ significantly")
print(f"  B alone vs B after A: check if scores differ significantly")
print("  If scores change based on context, position bias is present.")

## Step 3: Detect verbosity bias

**What is verbosity bias?** The judge prefers longer responses, even when the extra length adds zero information. A 500-character response with filler scores higher than a 70-character response with the same facts.

**How we test it:** Two responses with the **exact same factual content**:
- **Concise** (70 chars): `"BA117: JFK 7PM to LHR 7AM, $450. DL1: JFK 9:30PM to LHR 9:30AM, $520."`
- **Verbose** (500+ chars): Same two flights, but wrapped in "Great question! I'd be happy to help..." filler.

**What to look for:** If the verbose response scores higher despite having the same facts, verbosity bias is present. The score difference tells you the magnitude.

In [ ]:
"""Verbosity bias test: same info, different lengths."""

# Same factual content, but the verbose version adds filler
CONCISE = "BA117: JFK 7PM to LHR 7AM, $450. DL1: JFK 9:30PM to LHR 9:30AM, $520."

VERBOSE = (
    "Great question! I'd be happy to help you find flights from New York City "
    "to London for next Friday. After searching through available options, "
    "I found two excellent choices for you to consider.\n\n"
    "The first option is British Airways flight BA117, which departs from "
    "John F. Kennedy International Airport at 7:00 PM and arrives at London "
    "Heathrow Airport at 7:00 AM the following morning. This flight is priced "
    "at $450 per person.\n\n"
    "The second option is Delta Air Lines flight DL1, departing JFK at 9:30 PM "
    "and arriving at LHR at 9:30 AM. This flight costs $520 per person.\n\n"
    "Both are excellent options! Let me know if you need more details."
)

cases_verbosity = [
    Case(name="concise", input="Find flights NYC to London"),
    Case(name="verbose", input="Find flights NYC to London"),
]

responses_verbosity = {"concise": CONCISE, "verbose": VERBOSE}

experiment = Experiment(cases=cases_verbosity, evaluators=[evaluator])
reports = experiment.run_evaluations(lambda case: responses_verbosity[case.name])
reports[0].display()

print(f"\nConcise: {len(CONCISE)} chars")
print(f"Verbose: {len(VERBOSE)} chars ({len(VERBOSE)/len(CONCISE):.1f}x longer)")
print("Same facts, different length. If scores differ, verbosity bias is present.")

In [ ]:
import matplotlib.pyplot as plt

# Extract scores from the verbosity bias reports
concise_score = None
verbose_score = None
for case_result in reports[0].cases:
    if case_result["case_name"] == "concise":
        concise_score = case_result.get("score", 0)
    elif case_result["case_name"] == "verbose":
        verbose_score = case_result.get("score", 0)

fig, ax = plt.subplots(figsize=(10, 5))
fig.set_facecolor('white')

labels = ['Concise', 'Verbose']
scores = [concise_score, verbose_score]
colors = ['#FF7043', '#42A5F5']

bars = ax.bar(labels, scores, color=colors, width=0.5, edgecolor='white')

for bar, score in zip(bars, scores):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
            f'{score:.2f}', ha='center', va='bottom', fontsize=14, fontweight='bold')

ax.set_ylabel('Score', fontsize=12)
ax.set_title('Verbosity Bias: Concise vs Verbose Scores\n(Same facts, different length)', fontweight='bold', fontsize=14)
ax.set_ylim(0, 1.1)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

## Step 4: Mitigate verbosity bias with rubric instructions

If Step 3 showed verbosity bias, we can reduce it by **adding explicit anti-bias instructions to the rubric**. This tells the judge to ignore length and focus on information content.

**The change:** We add this paragraph to the rubric:
> "IMPORTANT: Do NOT reward response length. A concise response with all required facts should score the same as a verbose response with the same facts. Score based on information content only."

**What to look for:** Run the same concise vs. verbose test. The scores should be **closer together** than in Step 3. The anti-bias instruction does not eliminate bias completely, but reduces it.

In [ ]:
"""Mitigate verbosity bias with an improved rubric."""

# Rubric with explicit anti-verbosity instruction
debiased_evaluator = OutputEvaluator(
    rubric=(
        "Rate helpfulness 0-1. A helpful response lists specific flights "
        "with airline, time, and price.\n\n"
        "IMPORTANT: Do NOT reward response length. A concise response with "
        "all required facts should score the same as a verbose response with "
        "the same facts. Score based on information content only."
    ),
    model=MODEL,
)

experiment = Experiment(cases=cases_verbosity, evaluators=[debiased_evaluator])
reports = experiment.run_evaluations(lambda case: responses_verbosity[case.name])
reports[0].display()

print("\nWith anti-verbosity rubric, scores should be closer between concise and verbose.")

## Key Takeaways

1. **LLM judges have systematic biases.** Position bias and verbosity bias are the most common. Test for them before trusting evaluation results.

2. **Detection is straightforward.** Score the same content in different contexts (order, length). If scores change, bias is present.

3. **Rubric-level mitigation works.** Adding explicit anti-bias instructions to the rubric reduces (but does not eliminate) bias.

4. **For high-stakes evaluation, use multiple runs.** The [On Randomness paper](https://arxiv.org/abs/2602.07150) recommends running evaluations multiple times and reporting pass@k metrics.

**Next:** [Demo 03 - Multi-Judge Ensemble](../03-multi-judge-ensemble/) uses multiple judge models to further reduce bias.